[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dolev31/TabAgent/blob/main/notebooks/quickstart.ipynb)

# TabAgent in a few minutes

TabAgent replaces an LLM agent's repeated closed-set decisions, such as tool shortlisting or routing, with a
cheap classifier trained on the agent's own successful runs
([paper](https://arxiv.org/abs/2602.16429)). This notebook runs top to bottom on a free Colab CPU runtime:

1. Install
2. Your own decision table in seconds (offline)
3. A real benchmark: SkillsBench skill selection
4. Deploy: save, load and time one decision
5. Next steps

## 1. Install

TabAgent with the `embed` extra, which adds the e5 encoder's dependencies (PyTorch and Transformers).

In [ ]:
# Once TabAgent is on PyPI: %pip install -q "tabagent[embed]"
%pip install -q "tabagent[embed] @ git+https://github.com/dolev31/TabAgent"

## 2. Your own decision table in seconds (offline)

Log one row per decision with the candidates the agent chose. `tabagent.pointwise` expands the log to one row
per (decision, candidate) with a 0/1 label, and `TabAgentRanker` is a scikit-learn estimator on those rows.
The toy log below, 80 requests over six tools, stands in for yours.

In [ ]:
import numpy as np
import pandas as pd

import tabagent
from tabagent import TabAgentRanker

TOOLS = pd.DataFrame(
    {
        "tool": ["search_flights", "book_flight", "get_weather", "send_email", "read_inbox", "play_song"],
        "tool_description": [
            "search flights between two airports",
            "book a seat on a flight",
            "weather forecast for a city",
            "send an email message",
            "read the latest emails",
            "play a song or playlist",
        ],
    }
)
REQUESTS = [
    ("find a flight to {c} and book it", ["search_flights", "book_flight"]),
    ("what is the weather in {c} this weekend", ["get_weather"]),
    ("reply to the email about my trip to {c}", ["read_inbox", "send_email"]),
    ("play some music from {c}", ["play_song"]),
]
CITIES = ["rome", "oslo", "lima", "cairo"]
rng = np.random.default_rng(0)
decisions = pd.DataFrame(
    [
        {"task_id": i, "request": req.format(c=rng.choice(CITIES)), "tools_used": used}
        for i, (req, used) in enumerate(REQUESTS[rng.integers(len(REQUESTS))] for _ in range(80))
    ]
)
decisions.head(3)

In [ ]:
rows = tabagent.pointwise(decisions, chosen="tools_used", catalog=TOOLS, candidate_col="tool")
X, y = rows.drop(columns=["chosen"]), rows["chosen"]

# pair_embeddings=False keeps this section offline (no encoder download)
ranker = TabAgentRanker(group_col="task_id", candidate_col="tool", pair_embeddings=False)
ranker.fit(X, y)  # infers the feature card; picks TabSynth and the candidate prior by nested validation
ranker.selection_  # the recipes it tried

In [ ]:
new = pd.DataFrame({"task_id": 999, "request": "find a flight to paris and book it"}, index=[0])
new = new.merge(TOOLS, how="cross")  # the new decision: one row per candidate tool
ranker.rank(new, k=2)

In [ ]:
# grouped cross-validation against BM25 and chance, then latency and cost per decision on this CPU
print(ranker.report(X, y, n_splits=4))

This toy log is easy by construction. The next section uses a real benchmark.

## 3. A real benchmark: SkillsBench skill selection

SkillsBench pairs agent tasks with a catalog of skills, and each task ships a curated bundle of them. The
decision: which skills to hand the agent for a task. `datasets.load` rebuilds the decision table from the
pinned public release (3.5 MB, SHA-256 verified) and `datasets.card` returns the frozen feature card behind
the paper's result. The card's embedding feature downloads the e5-small-v2 encoder from Hugging Face on first
use (about 128 MB).

In [ ]:
from concurrent.futures import ThreadPoolExecutor

from tabagent import TabAgent, TabHead, datasets

NAME = "skillsbench/skill-select"

# Optional: fetch the release's small files in parallel (datasets.load fetches them one by one).
# Finished files are cached, so after a network error, run the cell again.
source = datasets.info(NAME).source
with ThreadPoolExecutor(8) as pool:
    list(pool.map(lambda path: source.path(path, download=True), source.files()))

table = datasets.load(NAME, download=True)  # one row per (task, skill); relevant = skill in the task's bundle
card = datasets.card(NAME)
print(f"{len(table.groups)} tasks x {table.df['skill_name'].nunique()} skills = {len(table)} rows")
pd.DataFrame(
    [(f.name, f.family, f.type, f.source) for f in card.features],
    columns=["feature", "family", "type", "source"],
)

The paper evaluates with five-fold task-level cross-validation over three seeds. For a quick look, fit
TabAgent on four of the paper's five folds and score the held-out fifth, with one seed.

In [ ]:
from tabagent.evaluation import PAPER_SEEDS, bootstrap_ci, per_decision

fold = datasets.folds(NAME)  # the paper's task-level split: {task: fold}
train = table.subset([t for t in table.groups if fold[t] != 0])
test = table.subset([t for t in table.groups if fold[t] == 0])

agent = TabAgent(card, head=TabHead("ranking"), seed=PAPER_SEEDS[0])  # CatBoost, the paper's configuration
agent.fit(train)  # the first fit downloads the e5 encoder
scores = agent.score(test.drop_labels())  # held-out tasks, labels removed

print(f"trained on {len(train.groups)} tasks, scored {len(test.groups)} held-out tasks (one fold, one seed)")
for metric, label in [("p@r", "P@R"), ("recall@9", "Recall@9")]:
    ci = bootstrap_ci(per_decision(test, scores, metric))  # 95% bootstrap interval over the held-out tasks
    print(f"{label:9} {100 * ci.estimate:5.1f}   95% CI [{100 * ci.low:.1f}, {100 * ci.high:.1f}]")

One held-out fold is a quick look, not the paper's protocol, and its wide interval shows how noisy it is.
The paper's full protocol (all five folds, three seeds) runs from a clone of the repository with
`python -m experiments.reproduce --cells skill-selection` (add `--download` on the first run). The paper
reports **P@R 80** and **Recall@9 91**.

The five skills TabAgent hands the agent for one held-out task, next to the task's curated bundle (the
label):

In [ ]:
task = test.subset([test.groups[0]])
bundle = sorted(task.df.loc[task.labels == 1, "skill_name"])
top5 = agent.decide(task.drop_labels(), k=5)
print("task:          ", task.df["task_name"].iloc[0])
print("curated bundle:", bundle)
top5.assign(in_bundle=top5["candidate"].isin(bundle))[["candidate", "score", "in_bundle"]]

## 4. Deploy

`save` writes a portable, pickle-free artifact; `tabagent.load` restores it and checks every file against the
artifact's SHA-256 manifest. The cell reloads the head, decides for the held-out task above and times that
decision on this runtime's CPU.

In [ ]:
import tempfile
import time

path = agent.save(tempfile.mkdtemp(prefix="tabagent_skills_"))
head = tabagent.load(path)
assert np.allclose(head.score(test.drop_labels()), scores)  # the loaded head scores as before

candidates = task.drop_labels()  # the held-out task above: one row per candidate skill
latency_ms = []
for _ in range(20):
    start = time.perf_counter()
    shortlist = head.decide(candidates, k=5)
    latency_ms.append(1000 * (time.perf_counter() - start))
print(
    f"Measured in this notebook: {np.median(latency_ms):.1f} ms per decision over {len(candidates)} skills "
    "on this CPU (median of 20 calls, e5 text embeddings already cached)"
)
shortlist[["candidate", "score"]]

This timing is this runtime's, not the paper's: the paper's 2.682 ms per decision was measured on another
decision, AppWorld tool shortlisting. From agent logs, `tabagent.onboard` builds a head that decides from the
raw trajectory, and `tabagent serve` puts it behind HTTP; see the
[deployment guide](https://dolev31.github.io/TabAgent/deployment/).

## 5. Next steps

- [Documentation](https://dolev31.github.io/TabAgent/): concepts, components, datasets and deployment.
- [Onboard an agent](https://dolev31.github.io/TabAgent/onboarding/): build a head from your own agent's logs
  with an LLM key.
- [Reproduce the paper](https://dolev31.github.io/TabAgent/reproducing/): every public result, its command and
  the released value.
- [The paper on arXiv](https://arxiv.org/abs/2602.16429) and the
  [code on GitHub](https://github.com/dolev31/TabAgent).

If you use TabAgent, please cite the paper:

```bibtex
@misc{levy2026tabagent,
  title  = {TabAgent: Distilling LLM Agent Components into Cheap Classifiers},
  author = {Levy, Ido and Shlomov, Segev and Yaeli, Avi and Mashkif, Nir and Adi, Asaf and
            Goldstein, Yinon and Shapira, Eilam and Choshen, Leshem},
  year   = {2026},
  eprint = {2602.16429},
  archivePrefix = {arXiv},
  primaryClass  = {cs.CL},
  url    = {https://arxiv.org/abs/2602.16429}
}
```